In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

save_path = r'D:\Projects\primary-vs-secondary\notebooks\resources\12\wnir.html'

In [3]:
df = pd.read_parquet(
    r"D:\Projects\primary-vs-secondary\data\interim\04_wnir.parquet"
).copy()
df_primary = df[df["market_type"] == "primary"]
df_secondary = df[df["market_type"] == "secondary"]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111656 entries, 0 to 2111655
Data columns (total 25 columns):
 #   Column                             Dtype         
---  ------                             -----         
 0   address                            string        
 1   longitude                          float32       
 2   latitude                           float32       
 3   area                               float32       
 4   room_count                         uint8         
 5   floor                              int16         
 6   market_type                        category      
 7   flat_type                          category      
 8   build_year                         uint16        
 9   balcony                            bool          
 10  price                              float32       
 11  price_per_square_meter             float32       
 12  date                               datetime64[ns]
 13  price_per_square_meter_normalized  float32       
 14  pr

In [7]:
df_primary.head()

,address,longitude,latitude,area,room_count,floor,market_type,flat_type,build_year,balcony,...,wnir_primary_500,neighbours_count_primary_500,wnir_secondary_500,neighbours_count_secondary_500,wnir_ratio_500,wnir_primary_1000,neighbours_count_primary_1000,wnir_secondary_1000,neighbours_count_secondary_1000,wnir_ratio_1000
0,"1-й Дубровский пр-д, вл. 78/14, к. 1.4",37.683254,55.724457,58.310001,2,16,primary,flat,2015,False,...,331507.75000,1079.0,525785.37500,694.0,0.630500,340383.46875,4956.0,518458.09375,4222.0,0.656530
1,"Складочная ул., вл. 1, к. С1",37.591412,55.803158,67.199997,1,13,primary,flat,2015,False,...,443387.03125,2390.0,500991.28125,124.0,0.885019,445348.71875,4629.0,474654.40625,1803.0,0.938259
2,"Волоколамское шоссе, вл. 67, кв-л 1",37.429195,55.811207,37.900002,1,9,primary,flat,2015,False,...,471386.62500,4639.0,532411.31250,67.0,0.885381,471228.31250,8151.0,626805.18750,1112.0,0.751794
3,"Волоколамское шоссе, вл. 67, кв-л 1",37.429195,55.811207,87.900002,3,12,primary,flat,2015,False,...,471386.62500,4639.0,532411.31250,67.0,0.885381,471228.31250,8151.0,626805.18750,1112.0,0.751794
4,"Складочная ул., вл. 1, к. С1",37.591412,55.803158,67.199997,1,21,primary,flat,2015,False,...,443387.03125,2390.0,500991.28125,124.0,0.885019,445348.71875,4629.0,474654.40625,1803.0,0.938259


In [18]:
df_primary[df_primary['neighbours_count_secondary_1000'] == 0].shape

(333431, 25)

In [9]:
import numpy as np
import pandas as pd
import folium
from branca.colormap import linear


def make_grid_map_primary_only(
    df_primary,
    column_name: str,
    save_path: str,
    grid_size: int = 200,
    min_count: int = 1
):

    df = df_primary[["latitude", "longitude", column_name]].dropna().copy()
    df["value"] = df[column_name]

    lat_min, lat_max = df["latitude"].min(), df["latitude"].max()
    lon_min, lon_max = df["longitude"].min(), df["longitude"].max()

    lat_bins = np.linspace(lat_min, lat_max, grid_size)
    lon_bins = np.linspace(lon_min, lon_max, grid_size)

    def assign_bins(df):
        df = df.copy()
        df["lat_bin"] = np.digitize(df["latitude"], lat_bins)
        df["lon_bin"] = np.digitize(df["longitude"], lon_bins)
        return df

    df = assign_bins(df)

    grouped = (
        df.groupby(["lat_bin", "lon_bin"])
        .agg(
            value_mean=("value", "mean"),
            value_median=("value", "median"),
            count=("value", "count"),
        )
        .reset_index()
    )

    grouped = grouped[grouped["count"] >= min_count]

    center = [(lat_min + lat_max) / 2, (lon_min + lon_max) / 2]
    m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")

    cm = linear.YlOrRd_09.scale(df["value"].min(), df["value"].max())
    cm.caption = column_name
    cm.add_to(m)

    def draw_layer(value_col, name):

        fg = folium.FeatureGroup(name=name, show=False)

        for _, row in grouped.iterrows():
            lat_idx = int(row["lat_bin"])
            lon_idx = int(row["lon_bin"])

            if lat_idx <= 0 or lon_idx <= 0:
                continue

            if lat_idx >= len(lat_bins) or lon_idx >= len(lon_bins):
                continue

            lat0 = lat_bins[lat_idx - 1]
            lat1 = lat_bins[lat_idx]
            lon0 = lon_bins[lon_idx - 1]
            lon1 = lon_bins[lon_idx]

            folium.Rectangle(
                bounds=[[lat0, lon0], [lat1, lon1]],
                fill=True,
                fill_color=cm(row[value_col]),
                fill_opacity=min(0.8, 0.3 + row["count"] / 50),
                weight=0,
                tooltip=f"""
                {name}<br>
                Value: {row[value_col]:.4f}<br>
                Count: {row['count']}
                """,
            ).add_to(fg)

        return fg

    layers = [
        draw_layer("value_mean", "Mean"),
        draw_layer("value_median", "Median"),
    ]

    layers[0].add_to(m)

    for layer in layers:
        layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    m.save(save_path)

In [14]:
make_grid_map_primary_only(df_primary, 'wnir_ratio_1000', save_path=save_path)